# Nano Relation Extractor

Trains a multilingual entity and relation extractor, exports it to ONNX,
quantises it to INT8 and packages the result. One stage per cell.

Runs anywhere. Locally it uses the checkout it was started from; on Kaggle or
Colab it clones the repository and installs what the base image lacks.

**On Kaggle**, open Settings before running:

- Accelerator: **GPU T4 x2**. Turing has float16 tensor cores that the P100
  lacks, and both cards are used.
- Internet: **on**. The corpora are fetched at run time.

In [ ]:
import importlib.util, os, pathlib, subprocess, sys

REPO = "https://github.com/apptivitypl/nano-relation-extractor"

if importlib.util.find_spec("nano_re") is None:
    on_kaggle = pathlib.Path("/kaggle").exists()
    local_src = pathlib.Path.cwd().parent / "src"

    if local_src.joinpath("nano_re").is_dir():
        sys.path.insert(0, str(local_src))
    elif on_kaggle:
        for name in ("HF_HOME", "HF_DATASETS_CACHE", "TRANSFORMERS_CACHE"):
            os.environ[name] = "/kaggle/temp/hf"
        pathlib.Path("/kaggle/temp/hf").mkdir(parents=True, exist_ok=True)

        checkout = pathlib.Path("/kaggle/working/nano-relation-extractor")
        if not checkout.exists():
            subprocess.run(
                ["git", "clone", "--depth", "1", REPO, str(checkout)], check=True
            )
        missing = [
            name
            for name in ("onnxruntime", "onnxscript")
            if importlib.util.find_spec(name) is None
        ]
        if missing:
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "-q", *missing], check=True
            )
        sys.path.insert(0, str(checkout / "src"))
        os.chdir(checkout)
    else:
        raise SystemExit(
            "nano_re is not importable. Run 'uv sync' in the project root, "
            "then start Jupyter with 'uv run jupyter lab'."
        )

import torch

import nano_re

print("nano_re", nano_re.__version__)
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    properties = torch.cuda.get_device_properties(0)
    print(f"gpu {properties.name} {properties.total_memory / 1e9:.0f} GB")

## Scale of the run

`LIMIT` caps documents per corpus and `EPOCHS` how many passes to make. Only the
documents the run needs are downloaded, so `LIMIT` governs disk, network and
time together.

Measured on a Kaggle T4, eight languages, sequence length 384:

| `LIMIT` | Documents per epoch | 3 epochs | 5 epochs |
| --- | --- | --- | --- |
| 5000 | 13,053 | 14 min | 24 min |
| 20000 | ~37,000 | 40 min | 65 min |
| 60000 | ~77,000 | 85 min | 2.3 h |

Add about ten minutes for export, quantisation and the benchmark. Kaggle
sessions stop at twelve hours, so there is room to go well past these.

The first run reached NER F1 0.58 and relation F1 0.37 in three epochs, with
both still improving, so the model was not saturated: more of either dial should
help. Everything else configures itself from the detected device and the corpora.

In [ ]:
import os
from dataclasses import replace

from nano_re.config import PipelineConfig
from nano_re.pipeline import Pipeline

LANGUAGES = "pl,en,de,fr,es,it,nl,pt"
LIMIT = 20000
EPOCHS = 5

os.environ.setdefault("NANO_RE_LANGUAGES", LANGUAGES)
os.environ.setdefault("NANO_RE_MAX_SEQUENCE_LENGTH", "384")

config = PipelineConfig.from_env()
config = config.with_overrides(
    data=replace(config.data, limit=LIMIT),
    training=replace(config.training, epochs=EPOCHS),
)
pipeline = Pipeline(config)

print("encoder:  ", config.model.backbone_name)
print("languages:", ", ".join(config.data.languages))
print("documents:", f"{LIMIT} per corpus, {EPOCHS} epochs")
print("artifacts:", config.artifacts_dir.resolve())

## Corpora and label schema

Downloads the corpora, interleaves them by weight and derives the label schema
from what was actually read. Which relations exist depends on which languages
are in scope, so the inventory is counted rather than declared, and its long
tail is dropped by coverage.

In [ ]:
schema = pipeline.prepare()

print("Entity types:", ", ".join(schema.entity_types))
print("BIO tags:    ", schema.num_bio_labels)
print("Relations:   ", schema.num_relation_labels)

counts = pipeline.data_module.inventory.counts
top = sorted(counts.items(), key=lambda item: -item[1])[:10]
for relation_id, count in top:
    print(f"  {relation_id:<8} {count:>6}  {schema.describe_relation(relation_id)}")

In [ ]:
bundle = pipeline.data_module.build_corpus(config.data.train_split, training=True)
print(bundle.describe())

for index in range(len(bundle.dataset)):
    sample = bundle.dataset[index]
    if sample is None:
        continue
    print(f"\nFirst usable document: {sample.doc_id}")
    print(f"  sub-words {sample.input_ids.shape[0]}, entities {sample.num_entities},"
          f" candidate pairs {sample.num_pairs}")
    print(f"  relation supervision: {sample.has_relation_supervision}")
    print(f"  mention mask rows sum to one: "
          f"{[round(float(x), 3) for x in sample.mention_mask.sum(-1)[:4]]}")
    break

## Training

Both heads share one encoder and train together under
`L = alpha * L_NER + beta * L_RE`. A corpus that annotates entities but not
relations is masked out of the relation term, so it cannot teach the relation
head that every pair is unrelated.

Before the first step, one real batch is attempted and halved until it fits, so
an out of memory failure is decided in seconds rather than hours in. Progress is
reported per batch with a rate and an estimate.

In [ ]:
training_report = pipeline.train()

best = training_report.best_evaluation
print(f"Best epoch:          {training_report.best_epoch}")
print(f"NER micro F1:        {best.ner.f1:.4f}")
print(f"Relation micro F1:   {best.relation.f1:.4f}")
print(f"Recall ceiling:      {best.relation_recall_ceiling:.4f}")

## Export and quantisation

The exporter compares the graph against PyTorch on three differently shaped
batches and fails if the relative deviation exceeds tolerance, or if the two
implementations would ever choose different classes.

Quantisation then tries several configurations and keeps the first whose
predictions still agree with float32, measured on real documents. It has to:
ONNX Runtime's dynamic path pairs unsigned activations with signed weights, and
on x86 without VNNI that product saturates, which on the first run collapsed
NER F1 from 0.63 to 0.005. If no configuration survives, the cell says so and
`model.onnx` is what you deploy.

In [ ]:
artifacts = pipeline.export()
quantisation = artifacts.quantization

print("Backend:            ", artifacts.export.exporter)
print("Dynamic shapes:     ", artifacts.export.dynamic_shapes_verified)
print(f"Relative deviation:  {artifacts.export.max_relative_deviation:.2e}")
print("Decisions match:    ", artifacts.export.decisions_match)
print()
print("Quantisation recipe:", quantisation.recipe)
if quantisation.agreement is not None:
    print(f"Agreement with FP32: {quantisation.agreement:.1%}")
print("Usable:             ", quantisation.is_usable)
for name, score in quantisation.rejected:
    print(f"  rejected {name}: {score:.1%}")
print(f"Size: {quantisation.source_bytes / 1e6:.1f} MB -> "
      f"{quantisation.target_bytes / 1e6:.1f} MB "
      f"({quantisation.compression_ratio:.2f}x)")

In [ ]:
benchmark = pipeline.benchmark(measure_accuracy=True)

print(f"FP32: {benchmark.fp32.median_ms:6.2f} ms/page  {benchmark.fp32.size_mb:7.1f} MB")
print(f"INT8: {benchmark.int8.median_ms:6.2f} ms/page  {benchmark.int8.size_mb:7.1f} MB")
print(f"Speedup {benchmark.speedup:.2f}x, size reduction {benchmark.size_reduction:.1%}")
print(f"F1 change from quantisation: NER {benchmark.ner_f1_delta:+.4f}, "
      f"relation {benchmark.relation_f1_delta:+.4f}")

## Bundle

Writes the model card from the measurements above, inventories the directory and
checks that nothing expected is missing.

In [ ]:
report = pipeline.package(
    training=training_report,
    benchmark=benchmark,
    quantization=artifacts.quantization,
)
print(report.render())
print("\nComplete:", report.is_complete)

## Using the model

Text of any length works: it is split into overlapping windows and the results
merged. Structured identifiers such as tax numbers and IBANs are matched by rule
and verified by checksum alongside whatever the model predicts.

In [ ]:
from nano_re.inference import RelationExtractor

extractor = RelationExtractor.from_bundle(
    pipeline.artifacts_dir, backend="onnx-int8", config=config
)
print("Backend:", extractor.backend_name, "\n")

text = (
    "Skai TV is a Greek free-to-air television network based in Piraeus. "
    "Skai TV is part of Skai Group, one of the largest media groups in the country."
)
print(extractor.extract(text).render())

In [ ]:
print((pipeline.artifacts_dir / "MODEL_CARD.md").read_text(encoding="utf-8"))

## Trimming the bundle

The float32 graph exists only for the benchmark comparison. Remove it if you are
near a storage limit, such as Kaggle's output quota.

In [ ]:
import pathlib

graph = pipeline.artifacts_dir / "model.onnx"
if graph.exists():
    print(f"removing the {graph.stat().st_size / 1e6:.0f} MB float32 graph")
    graph.unlink()

total = sum(p.stat().st_size for p in pipeline.artifacts_dir.rglob("*") if p.is_file())
print(f"bundle: {total / 1e6:.0f} MB")